## 2 common mathod to create TEMP TABLE

## Define the schema manually

In [ ]:
CREATE TEMP TABLE temp_transaction(
    transaction_id BIGINT,
    customer_id BIGINT,
    amount NUMERIC(18, 2),
    status TEXT,
    transaction_date DATE
)

In [ ]:
INSERT INTO temp_transaction(
    transaction_id,
    customer_id,
    amount,
    status,
    transaction_date
)
VALUE(
    101,
    1,
    15000,
    'Success'
    '2026-11-15'
)

## Create a temporary table from a query

In [ ]:
CREATE TEMP TABLE temp_transaction AS
        SELECT transaction_id,
                customer_id,
                amount,
                status,
                transaction_date
        FROM raw_transactions
        WHERE status = 'Success'

### refence successful temp table above to Calculate total successful payment amount

In [ ]:
SELECT transaction_id
        SUM(amount) AS total_amount
    FROM temp_transaction
    ORDER BY customer_id

### Temporary tables behave like regular tables in many respects. You can modify their records without modifying the original source dataset.

In [ ]:
CREATE TEMP TABLE temp_working_transaction AS

SELECT * FROM raw_transaction

## insert new record to the above query

In [ ]:
INSERT INTO temp_working_transaction(
    transaction_id,
    customer_id,
    amount,
    status,
    transaction_date
)
VALUE(
    109,
    3,
    35000,
    'Success',
    2026-11-15
)

### update the existing record from the above reference

In [ ]:
UPDATE temp_working_transaction
SET status = 'Success'
WHERE transaction_id = 105

## Incremental extraction using a watermark

### Step 1 — Create an incremental temporary table

In [ ]:
CREATE TEMP TABLE temp_incremental_transaction AS

SELECT t.transaction_id,
        t.amount,
        t.updated_at
    FROM raw_payment_event AS r
    WHERE t.updated_at > (
        SELECT MAX(w.last_processed_at),
        FROM etl_watermarks AS w
        WHERE w.pipeline_name = 'payment_pipeline'
    )
    AND t.updated_at <= TIMESTAMP '2026-11-16 10:00:00';

### Step 2 — Validate the incremental batch

In [ ]:
SELECT COUNT(*) AS total_transaction,
    MIN(upadated_at) AS earliest_transactions,
    MAX(updated_at) AS latest_transaction
    SUM(amount) AS total_amount
FROM temp_incremental_transaction;